# Clase 5 · 00 · Preflight: datos y experimento

En esta clase pasamos de **procesar** datos a **entrenar modelos con ellos**, y lo hacemos con la disciplina de MLOps: cada entrenamiento queda registrado en **MLflow** con sus parámetros, métricas, gráficos y el modelo entrenado, para poder compararlo, reproducirlo y ponerlo en uso.

| Notebook | Tema de la teoría | Qué vas a hacer |
|---|---|---|
| `01_primer_run` | Tracking: experimentos, runs, parámetros, métricas, artefactos | Ver qué registra MLflow en un run hecho a mano y con autolog |
| `02_mis_experimentos` | Experimentación | **Tus propios runs**: elegís modelo, features e hiperparámetros con widgets |
| `03_comparar_y_elegir` | Comparación y selección | Comparar todos tus runs y elegir el mejor con la métrica correcta |
| `04_registro_y_uso` | Model Registry, alias, inferencia batch, monitoreo | Registrar el mejor modelo, usarlo para puntuar transacciones nuevas y controlarlo |

**El problema.** En las clases anteriores `is_fraud` era una **alerta generada por una regla fija** (importe alto o transferencias marcadas). Ahora el área de riesgo nos pasa qué transacciones terminaron en **contracargo confirmado** (`chargeback`): el cliente desconoció la compra y la plataforma perdió el dinero. Queremos un modelo que anticipe el contracargo con los datos que tenemos al momento de la compra.

Este notebook arma la tabla `ml_features` a partir de las tablas Silver de la clase 2 y crea tu experimento de MLflow.

In [ ]:
dbutils.widgets.text("student_id", "alumno01", "01 - Identificador")
dbutils.widgets.dropdown("scale", "small", ["test", "small", "demo"], "02 - Escala")

In [ ]:
%run ../common/config

In [ ]:
%run ../common/ml_lab

In [ ]:
from pyspark.sql import functions as F
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import mlflow
from mlflow.tracking import MlflowClient

config = CourseConfig(spark.sql("SELECT current_catalog()").first()[0],
                      dbutils.widgets.get("student_id"), dbutils.widgets.get("scale"))
create_course_namespace(spark, config)
spark.conf.set("spark.sql.session.timeZone", "UTC")
T = lambda name: qualified_table(config, name)
client = MlflowClient()

In [ ]:
tables = {name: T(name) for name in ["silver_customers", "silver_products", "silver_transactions"]}
for name, table in tables.items():
    assert spark.catalog.tableExists(table), f"Falta {table}. Ejecutá la clase 2 con el mismo student_id y escala."
counts = {name: spark.table(table).count() for name, table in tables.items()}
assert counts["silver_customers"] == config.rows["customers"], "student_id o scale no coinciden con las clases anteriores"
assert counts["silver_products"] == config.rows["products"], "student_id o scale no coinciden con las clases anteriores"
display(pd.DataFrame(counts.items(), columns=["tabla", "filas"]))

## 1. MLflow está disponible

Databricks trae MLflow instalado y conectado al workspace: no hay que levantar ningún servidor. Los runs se guardan en un **experimento**, que es una carpeta de tu workspace.

In [ ]:
import sklearn
experiment = setup_experiment(spark, config)
print(f"MLflow {mlflow.__version__} · scikit-learn {sklearn.__version__}")
print(f"Tu experimento: {experiment.name}")
print(f"Tracking URI: {mlflow.get_tracking_uri()}")

Para verlo en la interfaz: menú lateral **Experiments** (o **Machine Learning → Experiments**) y buscá `bigdata_<tu_id>_clase05`. Por ahora está vacío; se llena en los notebooks siguientes.

## 2. La tabla de features

Cada fila es una transacción de la carga inicial (`source_batch_id = 'initial'`), unida con su cliente y su producto. Se agregan variables derivadas y la etiqueta `chargeback`.

| Columna | Qué es |
|---|---|
| `amount`, `payment_channel` | Importe y medio de pago de la compra |
| `hour`, `is_night`, `day_of_week` | Momento de la compra (`is_night` = entre las 0 y las 6 h) |
| `account_age_days` | Antigüedad de la cuenta del cliente al momento de la compra |
| `segment`, `country` | Datos del cliente |
| `category`, `price`, `amount_to_price` | Datos del producto y relación entre lo pagado y su precio de lista |
| `device_customers` | Cuántos clientes distintos usaron el mismo dispositivo |
| `rule_alert` | La alerta por regla de las clases anteriores (`is_fraud`) |
| `chargeback` | **Etiqueta**: 1 si la compra terminó en contracargo |

El `split` es **temporal**: se entrena con lo que pasó del 1 al 5 de marzo (`train`), se evalúa con el 6 (`test`) y el 7 se reserva como **transacciones nuevas** (`nuevas`) que el modelo puntuará en el notebook 04, como si todavía no supiéramos el resultado.

In [ ]:
features = build_ml_features(spark, config)
summary = (features.groupBy("split")
           .agg(F.count("*").alias("filas"),
                F.min(F.to_date("event_ts")).alias("desde"), F.max(F.to_date("event_ts")).alias("hasta"),
                F.round(F.avg("chargeback") * 100, 2).alias("% contracargos"),
                F.round(F.avg("rule_alert") * 100, 2).alias("% alertas de la regla"))
           .orderBy(F.expr("CASE split WHEN 'train' THEN 1 WHEN 'test' THEN 2 ELSE 3 END")))
display(summary)
display(features.where("split = 'train'").limit(10))

**Para observar:** los contracargos son pocos (las clases están **desbalanceadas**) y la regla de las clases anteriores alerta sobre muchas compras que no terminan en contracargo, y deja pasar otras que sí. Esa es la oportunidad para un modelo.

## 3. Una primera mirada

¿Qué variables se relacionan con el contracargo? Comparamos la tasa de contracargos por grupos, sólo con los datos de entrenamiento.

In [ ]:
train_spark = features.where("split = 'train'")
fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))
by_amount = (train_spark.withColumn("tramo", (F.floor(F.col("amount") / 250) * 250).cast("int"))
             .groupBy("tramo").agg(F.avg("chargeback").alias("tasa")).orderBy("tramo").toPandas())
axes[0].bar(by_amount["tramo"].astype(str), by_amount["tasa"] * 100, color="#2563eb")
axes[0].set_title("Por importe (tramos de 250)")
axes[0].tick_params(axis="x", rotation=60)
for ax, column, title in [(axes[1], "payment_channel", "Por medio de pago"), (axes[2], "is_night", "De noche (1) vs. de día (0)")]:
    group = train_spark.groupBy(column).agg(F.avg("chargeback").alias("tasa")).orderBy(column).toPandas()
    ax.bar(group[column].astype(str), group["tasa"] * 100, color="#16a34a")
    ax.set_title(title)
for ax in axes:
    ax.set_ylabel("% contracargos")
fig.tight_layout()
plt.show()

Ninguna variable sola alcanza: el contracargo depende de **combinaciones** de señales y tiene una parte de azar, como en un caso real. Por eso en los notebooks siguientes vamos a probar distintos modelos y conjuntos de variables, y MLflow nos va a ayudar a no perder la cuenta.